In [1]:
import sys
!{sys.executable} -m pip install opencv-python
import cv2 as cv

A1 = cv.imread("C:/Users/oscar/Downloads/For_LAQ3_required_data/For_LAQ3_required_data/A1.jpg")
A2 = cv.imread("C:/Users/oscar/Downloads/For_LAQ3_required_data/For_LAQ3_required_data/A2.jpg")

B1 = cv.imread("C:/Users/oscar/Downloads/For_LAQ3_required_data/For_LAQ3_required_data/B1.jpg")
B2 = cv.imread("C:/Users/oscar/Downloads/For_LAQ3_required_data/For_LAQ3_required_data/B2.jpg")

C1 = cv.imread("C:/Users/oscar/Downloads/For_LAQ3_required_data/For_LAQ3_required_data/C1.jpg")
C2 = cv.imread("C:/Users/oscar/Downloads/For_LAQ3_required_data/For_LAQ3_required_data/C2.jpg")

D1 = cv.imread("C:/Users/oscar/Downloads/For_LAQ3_required_data/For_LAQ3_required_data/D1.jpg")
D2 = cv.imread("C:/Users/oscar/Downloads/For_LAQ3_required_data/For_LAQ3_required_data/D2.jpg")

In [2]:
import os
import cv2 as cv
from PIL import Image
import numpy as np

def tissue_percent(n):
    if (len(n.shape) == 3) and (n.shape[2] == 3):
        n_s = n[:, :, 0] + n[:, :, 1] + n[:, :, 2]
        mask_p = 100 - (np.count_nonzero(n_s) / (n_s.size * 100))
    else:
        mask_p = 100 - np.count_nonzero(n) / n.size * 100
    return 100 - mask_p

def fixed_cube(img, d=200, th=20, name="base"):
    cubes = dict()
    n = 0
    
    for x in range(0, img.shape[0] - d, d):
        for y in range(0, img.shape[1] - d, d):
            
            window = img[x:x + d, y:y + d, :]
            cont = tissue_percent(window)
            
            if cont <= th:
                cubes[name + str(n)] = window
                n += 1
    return cubes


def save_image(img, path):
    if img is not None:
        if img.dtype == "bool":
            img = img.astype("uint8") * 255
        elif img.dtype == "float64":
            img = (img * 255).astype("uint8")

        img = Image.fromarray(img)
        img.save(path)
    return


def save_tiles(cube, path):
    if not os.path.exists(path):
        os.makedirs(path)
    
    for z in cube:
        save_image(cube[z], os.path.join(path, z + ".jpg"))
    
    print("Done")

In [3]:
A_cube = {}
for i, img in enumerate([A1, A2]):
    A_cube.update(fixed_cube(img, d=200, th=20, name="A"+str(i)))

B_cube = {}
for i, img in enumerate([B1, B2]):
    B_cube.update(fixed_cube(img, d=200, th=20, name="B"+str(i)))

C_cube = {}
for i, img in enumerate([C1, C2]):
    C_cube.update(fixed_cube(img, d=200, th=20, name="C"+str(i)))

D_cube = {}
for i, img in enumerate([D1, D2]):
    D_cube.update(fixed_cube(img, d=200, th=20, name="D"+str(i)))

In [4]:
save_tiles(A_cube, "cubed_A")
save_tiles(B_cube, "cubed_B")
save_tiles(C_cube, "cubed_C")
save_tiles(D_cube, "cubed_D")

Done
Done
Done
Done


In [5]:
tiles = []
path = []
group = []

import sys
!{sys.executable} -m pip install pandas
import pandas as pd

for tile in A_cube:
    tiles.append(tile)
    path.append(os.path.join(os.getcwd(), "cubed_A", tile + ".jpg"))
    group.append("A")

for tile in B_cube:
    tiles.append(tile)
    path.append(os.path.join(os.getcwd(), "cubed_B", tile + ".jpg"))
    group.append("B")

dat_AB = pd.DataFrame(list(zip(tiles, path, group)),
               columns =["name", "path", "group"])

In [6]:
import sys
!{sys.executable} -m pip install scikit-learn
from sklearn.model_selection import train_test_split
train_AB, test_AB = train_test_split(dat_AB, test_size=0.1)

In [7]:
import sys
!{sys.executable} -m pip install tensorflow
from tensorflow.keras.preprocessing.image import ImageDataGenerator

gtrain = ImageDataGenerator(rescale=1.0/255.0,
                            vertical_flip=True,
                            horizontal_flip=True,
                            height_shift_range=0.2,
                            fill_mode="reflect",
                            rotation_range=90)

gtest = ImageDataGenerator(rescale=1.0/255.0)

dgtrain_AB = gtrain.flow_from_dataframe(train_AB,
                                         x_col="path", y_col="group",
                                         target_size=(200,200),
                                         class_mode="categorical",
                                         batch_size=10)

dgtest_AB = gtest.flow_from_dataframe(test_AB,
                                       x_col="path", y_col="group",
                                       target_size=(200,200),
                                       class_mode="categorical",
                                       batch_size=10)

Found 86 validated image filenames belonging to 2 classes.
Found 10 validated image filenames belonging to 2 classes.


In [8]:
def define_model():
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import BatchNormalization, Conv2D, MaxPooling2D, Dense, Dropout, Flatten
    from tensorflow.keras.optimizers import SGD

    model = Sequential()
    model.add(Conv2D(64, (3, 3), activation='relu',
                     kernel_initializer='he_uniform',
                     input_shape=(200, 200, 3)))
    
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization())
    model.add(Conv2D(128, (3,3), activation="relu"))
    model.add(MaxPooling2D((2,2)))
    model.add(BatchNormalization()) 
    model.add(Flatten())
    model.add(Dropout(0.3))
    model.add(Dense(100, activation='relu', kernel_initializer='he_uniform'))
    model.add(Dense(2, activation='softmax'))
    opt = SGD(learning_rate=0.01, momentum=0.9)
    model.compile(optimizer=opt,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    
    return model
model1 = define_model()
history1 = model1.fit(dgtrain_AB,
                      validation_data=dgtest_AB,
                      epochs=10)

C:\Users\oscar\miniconda3\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 5s 436ms/step - accuracy: 0.5930 - loss: 66.2083 - val_accuracy: 0.3000 - val_loss: 279.3079
Epoch 2/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 400ms/step - accuracy: 0.6047 - loss: 1045.5801 - val_accuracy: 0.3000 - val_loss: 808674.3125
Epoch 3/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 413ms/step - accuracy: 0.5233 - loss: 0.7287 - val_accuracy: 0.7000 - val_loss: 383588032.0000
Epoch 4/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 397ms/step - accuracy: 0.5233 - loss: 0.6929 - val_accuracy: 0.7000 - val_loss: 686474560.0000
Epoch 5/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 420ms/step - accuracy: 0.5233 - loss: 0.6922 - val_accuracy: 0.7000 - val_loss: 715464768.0000
Epoch 6/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 415ms/step - accuracy: 0.5233 - loss: 0.6929 - val_accuracy: 0.7000 - val_loss: 618398720.0000
Epoch 7/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 403ms/step - accuracy: 0.5233 - loss: 0.6921 - val_accuracy: 0.7000 - val_loss: 496004768.0000
Epoch 8/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 405ms/step - accuracy: 0.523

# 1.what is the training and validation accuracy for model 1?

In [15]:
train_acc1 = history1.history['accuracy'][-1]
val_acc1 = history1.history['val_accuracy'][-1]

print(train_acc1, val_acc1)

0.5232558250427246 0.699999988079071


Training accuracy : 0.5232558250427246
Validation accuracy : 0.699999988079071

In [10]:
tiles = []
path = []
group = []

for tile in C_cube:
    tiles.append(tile)
    path.append(os.path.join(os.getcwd(), "cubed_C", tile + ".jpg"))
    group.append("C")

for tile in D_cube:
    tiles.append(tile)
    path.append(os.path.join(os.getcwd(), "cubed_D", tile + ".jpg"))
    group.append("D")

dat_CD = pd.DataFrame(list(zip(tiles, path, group)),
               columns =["name", "path", "group"])

train_CD, test_CD = train_test_split(dat_CD, test_size=0.1)

dgtrain_CD = gtrain.flow_from_dataframe(train_CD,
                                         x_col="path", y_col="group",
                                         target_size=(200,200),
                                         class_mode="categorical",
                                         batch_size=10)

dgtest_CD = gtest.flow_from_dataframe(test_CD,
                                       x_col="path", y_col="group",
                                       target_size=(200,200),
                                       class_mode="categorical",
                                       batch_size=10)

model2 = define_model()

history2 = model2.fit(dgtrain_CD,
                      validation_data=dgtest_CD,
                      epochs=10)

Found 78 validated image filenames belonging to 2 classes.
Found 9 validated image filenames belonging to 2 classes.


C:\Users\oscar\miniconda3\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 440ms/step - accuracy: 0.9103 - loss: 0.3257 - val_accuracy: 0.2222 - val_loss: 1409.2960
Epoch 2/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 436ms/step - accuracy: 0.8846 - loss: 10.0167 - val_accuracy: 0.2222 - val_loss: 21291.5020
Epoch 3/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 467ms/step - accuracy: 0.9615 - loss: 37.9181 - val_accuracy: 0.2222 - val_loss: 41974.8203
Epoch 4/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 467ms/step - accuracy: 0.5256 - loss: 5827626496.0000 - val_accuracy: 0.7778 - val_loss: 11719454399239252700022439936.0000
Epoch 5/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 442ms/step - accuracy: 0.5128 - loss: 4512551149739936518145835008.0000 - val_accuracy: 0.2222 - val_loss: 0.7277
Epoch 6/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 452ms/step - accuracy: 0.4744 - loss: 0.6988 - val_accuracy: 0.2222 - val_loss: 0.7134
Epoch 7/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 473ms/step - accuracy: 0.4744 - loss: 0.6952 - val_accuracy: 0.2222 - val_loss: 0.7048
Epoch 8/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 49

# 2. what is the training and validation accuracy for model 2?

In [13]:
train_acc2 = history2.history['accuracy'][-1]
val_acc2 = history2.history['val_accuracy'][-1]

print(train_acc2, val_acc2)

0.5256410241127014 0.7777777910232544


Training accuracy : 0.5256410241127014
Validation accuracy : 0.7777777910232544

# 3. which off the models performed best on its dataset and suggest possible reasons for the differences in performance

Model 2 perform better than Model 1 because its validation accuracy was higher.
The reason for the difference is that the features in the C and D dataset were easier for the model to distinguish compare to A and B. The A and B images may have looked more similar to each other, so it's harder to learn differences. Also, the patches extracted from A and B contaoned less informative tissue or more noise, which would make learning more difficult. Overall, Model 2 seems to have been trained on data that was more consistent or easier to seperate, which helped it achieve better performance.

# 4. test model 1 on test set 2 and model 2 on test set 1. Report the results

In [14]:
acc_1_2 = model1.evaluate(dgtest_CD)[1]
acc_2_1 = model2.evaluate(dgtest_AB)[1]

print(acc_1_2, acc_2_1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.7778 - loss: 71522320.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.7000 - loss: 0.6794
0.7777777910232544 0.699999988079071


Model 1 on test set 2 : 0.7777777910232544
Model 2 on test set 1 : 0.699999988079071

# 5. what are the possible explanations for the observations made in question 4 above.

The results show that the models were able to generalize, since they still achieved reasonable accuracy on a different dataset. This suggests that they learned features that are not entirely dataset-specific. However, the drop in performance indicates that the datasets are still different in ways that matter. Small changes in  texture or tissue structure affected how well the models could apply what they learned. Overall, the models learned useful patterns, but not strong enough to work equally well across all datasets.